In [ ]:
#| default_exp transformations

# Transformations

In [ ]:
#| hide
from fastcore.docments import docments, DocmentTbl
from nbdev.showdoc import *

In [ ]:
#| export

from __future__ import annotations
from typing import List, Dict, Optional, Callable, Tuple, Any, Union

from pyexpat import model
import pandas as pd
import numpy as np
from numba import jit
from scipy.stats import boxcox
from scipy.special import inv_boxcox
from numba import jit
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

#------------------------------------------------------------------------------
# Box-Cox Transformation Utility Functions
#------------------------------------------------------------------------------

def box_cox_transform(x,
                      shift: bool = False,
                      box_cox_lmda: float = None
                      ):
    """
    Applies a Box-Cox transformation to a series x.

    Parameters
    ----------
    x :array-like:
        The input data to be transformed.
    shift :bool
        Whether to shift the data by 1 before transformation to handle zeros.
    box_cox_lmda :float or None:
        The lambda parameter for the Box-Cox transformation. If None, it will be estimated from the data.

    Returns
    -------
    transformed_data and lmbda:
        The Box-Cox transformed data and the lambda used for transformation.
    """
    if (box_cox_lmda == None):
        if shift ==True:
            transformed_data, lmbda = boxcox((np.array(x)+1))
        else:
            transformed_data, lmbda = boxcox(np.array(x))
            
    if (box_cox_lmda != None):
        if shift ==True:
            lmbda = box_cox_lmda
            transformed_data = boxcox((np.array(x)+1), lmbda)
        else:
            lmbda = box_cox_lmda
            transformed_data = boxcox(np.array(x), lmbda)
    return transformed_data, lmbda

In [ ]:
#| echo: false
docments(box_cox_transform, full=True)
DocmentTbl(box_cox_transform)

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| x |  |  | The input data to be transformed. |
| shift | bool | False | Whether to shift the data by 1 before transformation to handle zeros. |
| box_cox_lmda | float | None | The lambda parameter for the Box-Cox transformation. If None, it will be estimated from the data. |
| **Returns** |  |  | **The Box-Cox transformed data and the lambda used for transformation.** |

In [ ]:
#| export

def back_box_cox_transform(y_pred: np.ndarray,
                           lmda: float,
                           shift: bool = False,
                           box_cox_biasadj: bool = False
                           ) -> np.ndarray:
    """
    Inverse Box-Cox transform.

    Parameters
    ----------
    y_pred : array-like
        The Box-Cox transformed forecast to be back-transformed.
    lmda : float
        The lambda parameter used in the Box-Cox transformation.
    shift : bool
        Whether the original data was shifted by 1 before transformation.
    box_cox_biasadj : bool
        Whether to apply bias adjustment to the back-transformed forecast.
    
    Returns
    -------
    forecast : array-like
        The back-transformed forecast.
    """
    if (box_cox_biasadj==False):
        if shift == True:
            forecast = inv_boxcox(y_pred, lmda)-1
        else:
            forecast = inv_boxcox(y_pred, lmda)
            
    if (box_cox_biasadj== True):
        pred_var = np.var(y_pred)
        if shift == True:
            if lmda ==0:
                forecast = np.exp(y_pred)*(1+pred_var/2)-1
            else:
                forecast = ((lmda*y_pred+1)**(1/lmda))*(1+((1-lmda)*pred_var)/(2*((lmda*y_pred+1)**2)))-1
        else:
            if lmda ==0:
                forecast = np.exp(y_pred)*(1+pred_var/2)
            else:
                forecast = ((lmda*y_pred+1)**(1/lmda))*(1+((1-lmda)*pred_var)/(2*((lmda*y_pred+1)**2)))
    return forecast

In [ ]:
#| echo: false
docments(back_box_cox_transform, full=True)
DocmentTbl(back_box_cox_transform)

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| y_pred | np.ndarray |  | The Box-Cox transformed forecast to be back-transformed. |
| lmda | float |  | The lambda parameter used in the Box-Cox transformation. |
| shift | bool | False | Whether the original data was shifted by 1 before transformation. |
| box_cox_biasadj | bool | False | Whether to apply bias adjustment to the back-transformed forecast. |
| **Returns** | **np.ndarray** |  | **The back-transformed forecast.** |

In [ ]:
#| hide

#------------------------------------------------------------------------------
# Croston's Method metrics
#------------------------------------------------------------------------------

# @jit(nopython=True)
def nzInterval(data, lag=0):
    """
    Computes the intervals between non-zero values in a time series.

    Args:
        data (array-like): Input time series data.
        lag (int): Number of lags to consider. Default is 0.

    Returns:
        numpy.ndarray: An array of intervals between non-zero values, with NaNs for the first `lag` values.
    """
    intervals = []
    last_nonzero_times = []
    if lag !=0:
        arr = data[:-lag]
    else:
        arr = data

    for i, j in enumerate(arr):
        if j > 0.5:
            last_nonzero_times.append(i)
        
        if len(last_nonzero_times)==1:
            intervals.append(1)
        else:
            inter = last_nonzero_times[-1]-last_nonzero_times[-2]
            intervals.append(inter)
    
    if lag !=0:
        nas = list(np.repeat(np.nan, lag))
        intervals = nas+intervals
    return np.array(intervals)

# @jit(nopython=True)
def zeroCumulative(data, lag=0):
    """
    Computes the cumulative count of consecutive zeros in a time series.
    Args:
        data (array-like): Input time series data.
        lag (int): Number of lags to consider. Default is 0.
    Returns:
        numpy.ndarray: An array of cumulative counts of consecutive zeros, with NaNs for the first `lag` values.
    """
    if lag !=0:
        arr = data[:-lag]
    else:
        arr = data

    count = 0
    result = []
    
    for value in arr:
        if value < 0.5:
            count += 1
        else:
            count = 0
        result.append(count)

    if lag !=0:
        nas = list(np.repeat(np.nan, lag))
        result = nas+result
    return np.array(result)


#------------------------------------------------------------------------------
# Target Encoding Utility Functions
#------------------------------------------------------------------------------

def kfold_target_encoder(df, feature_col, target_col, n_splits=5):
    """
    Perform K-fold target encoding for a given feature.

    Args:
        df (pd.DataFrame): Input dataframe.
        feature_col (str): Column name of the categorical feature.
        target_col (str): Target column name.
        n_splits (int): Number of folds for target encoding.

    Returns:
        np.array: Encoded target values.
    """
    df_encoded = df.copy()
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    encoded_col_name = f"{feature_col}_target_encoded"
    df_encoded[encoded_col_name] = np.nan
    feat_idx = df_encoded.columns.get_loc(feature_col)
    encode_idx = df_encoded.columns.get_loc(encoded_col_name)
    # Perform KFold encoding
    for train_idx, val_idx in kf.split(df):
        # Calculate target mean per category using only the training fold
        target_means = df.iloc[train_idx].groupby(feature_col)[target_col].mean()
        # Map means to the validation fold
        df_encoded.iloc[val_idx, encode_idx] = df_encoded.iloc[val_idx, feat_idx].map(target_means)
    # Fill missing values with overall target mean
    overall_mean = df[target_col].mean()
    df_encoded[encoded_col_name].fillna(overall_mean, inplace=True)
    return df_encoded[encoded_col_name].values

def target_encoder_for_test(train_df, test_df, feature_col):
    """
    Apply target encoding to the test set based on the training set.

    Args:
        train_df (pd.DataFrame): Training dataframe with encoded feature.
        test_df (pd.DataFrame): Test dataframe.
        feature_col (str): Column name to encode.

    Returns:
        np.array: Encoded values for the test set.
    """
    encoded_col_name = f"{feature_col}_target_encoded"
    target_means = train_df.groupby(feature_col)[encoded_col_name].mean()
    overall_mean = train_df[encoded_col_name].mean()
    test_encoded = test_df.copy()
    test_encoded[encoded_col_name] = test_df[feature_col].map(target_means)
    test_encoded[encoded_col_name].fillna(overall_mean, inplace=True)
    return test_encoded[encoded_col_name].values


In [ ]:
#| export

#------------------------------------------------------------------------------
# Transformation Utility Functions
#------------------------------------------------------------------------------

def fourier_terms(
    start_end_index: tuple,
    period: int,
    num_terms: int,
    frequency: str = "D"
) -> pd.DataFrame:
    """
    Generate Fourier terms for a given time index or range of indices.

    Parameters
    ----------
    start_end_index : tuple
        A tuple containing the start and end indices (can be integers or datetime strings).
    period : int
        The period of the seasonality (e.g., 7 for weekly, 12 for monthly).
    num_terms : int
        The number of Fourier terms to generate.
    frequency : str, optional
        The frequency of the time index (default is "D" for daily). For example, "D" for daily, "M" for monthly, "H" for hourly, etc. It is used when the start and end indices are datetime strings to create the appropriate date range.
    
    Returns
    -------
    pd.DataFrame
        A DataFrame containing the Fourier terms with the appropriate index.
    """
    start, end = start_end_index

    # Decide index type
    if isinstance(start, (int, np.integer)) and isinstance(end, (int, np.integer)):
        index = pd.RangeIndex(start, end+1)  # exclusive of end
        t = np.arange(len(index))
    else:
        # Convert to datetime if string given
        start, end = pd.to_datetime(start), pd.to_datetime(end)
        index = pd.date_range(start, end, freq=frequency)
        t = np.arange(len(index))

    # Build Fourier terms
    terms = {
        f'sin_{k}_{period}': np.sin(2 * np.pi * k * t / period)
        for k in range(1, num_terms + 1)
    }
    terms.update({
        f'cos_{k}_{period}': np.cos(2 * np.pi * k * t / period)
        for k in range(1, num_terms + 1)
    })

    return pd.DataFrame(terms, index=index)


In [ ]:
#| echo: false
docments(fourier_terms, full=True)
DocmentTbl(fourier_terms)

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| start_end_index | tuple |  | A tuple containing the start and end indices (can be integers or datetime strings). |
| period | int |  | The period of the seasonality (e.g., 7 for weekly, 12 for monthly). |
| num_terms | int |  | The number of Fourier terms to generate. |
| frequency | str | D | The frequency of the time index (default is "D" for daily). For example, "D" for daily, "M" for monthly, "H" for hourly, etc. It is used when the start and end indices are datetime strings to create the appropriate date range. |
| **Returns** | **pd.DataFrame** |  | **A DataFrame containing the Fourier terms with the appropriate index.** |

In [ ]:
#| export

class rolling_mean:
    def __init__(self,
                 window_size: int,
                 shift: int = 1,
                 min_samples: int = 1
                 ):

        """
        
        A class to compute the rolling mean of a time series with specified window size and shift.

        Parameters
        ----------
        window_size : int
            The size of the rolling window.
        shift : int, optional
            The number of periods to shift the data before applying the rolling mean (default is 1).
        min_samples : int, optional
            The minimum number of observations in the window required to have a value (default is 1).

        Returns
        -------
        None

        """
        self.shift = shift
        self.window_size = window_size
        self.min_samples = min_samples

    def __call__(self,
                 data: Union[pd.Series, np.ndarray],
                 is_forecast: bool = False
                 ) -> pd.Series:
        """
        Compute the rolling mean of the input data.
        
        Parameters
        ----------
        data : array-like
            The input time series data for which to compute the rolling mean.
        is_forecast : bool, optional
            Whether the data is a forecast (default is False). If True, the shift will be adjusted to align with the forecasted period. For example, if it's a forecast, for the forecasting next value, we might want to shift by one less than usual to align with the forecasted period

        Returns
        -------
        pd.Series
            A Series containing the rolling mean of the input data, shifted and computed according to the specified parameters.
        """

        if is_forecast:
            # For example, if it's a forecast, for the forecasting next value, we might want to shift by one less than usual to align with the forecasted period
            return pd.Series(data).shift(self.shift-1).rolling(self.window_size, min_periods=self.min_samples).mean()
        # If not a forecast, apply the usual shift
        else:
            # If not a forecast, apply the usual shift
            return pd.Series(data).shift(self.shift).rolling(self.window_size, min_periods=self.min_samples).mean()
        
    def get_name(self):
        return f"rolling_mean_{self.window_size}_{self.shift}"

In [ ]:
#| echo: false
docments(rolling_mean, full=True)
DocmentTbl(rolling_mean)

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| window_size | int |  | The size of the rolling window. |
| shift | int | 1 | The number of periods to shift the data before applying the rolling mean (default is 1). |
| min_samples | int | 1 | The minimum number of observations in the window required to have a value (default is 1). |
| **Returns** |  |  |  |

In [ ]:
#| export

class rolling_quantile:
    def __init__(self,
                 window_size: int,
                 quantile: float,
                 shift: int = 1,
                 min_samples: int = 1
                 ):

        """
        A class to compute the rolling quantile of a time series with specified window size, quantile, and shift.

        Parameters
        ----------
        window_size : int
            The size of the rolling window.
        quantile : float
            The quantile to compute (between 0 and 1).
        shift : int, optional
            The number of periods to shift the data before applying the rolling quantile (default is 1).
        min_samples : int, optional
            The minimum number of observations in the window required to have a value (default is 1).
        
        Returns
        -------
        None
        """
        self.shift = shift
        self.window_size = window_size
        self.quantile = quantile
        self.min_samples = min_samples

    def __call__(self,
                 data: Union[pd.Series, np.ndarray],
                 is_forecast: bool = False
                 ) -> pd.Series:
        """
        
        Compute the rolling quantile of the input data.
        
        Parameters:
        ----------
        data : array-like
            The input time series data for which to compute the rolling quantile.
        is_forecast : bool, optional
            Whether the data is a forecast (default is False). If True, the shift will be adjusted to align with the forecasted period. For example, if it's a forecast, for the forecasting next value, we might want to shift by one less than usual to align with the forecasted period
        
        Returns
        -------
        pd.Series
            A Series containing the rolling quantile of the input data, shifted and computed according to the specified parameters.
        """
        if is_forecast:
            return pd.Series(data).shift(self.shift-1).rolling(self.window_size, min_periods=self.min_samples).quantile(self.quantile)
        # If not a forecast, apply the usual shift
        else:
            # If not a forecast, apply the usual shift
            return pd.Series(data).shift(self.shift).rolling(self.window_size, min_periods=self.min_samples).quantile(self.quantile)
    def get_name(self):
        return f"rolling_quantile_{self.window_size}_{self.quantile}_{self.shift}"

In [ ]:
#| echo: false
docments(rolling_quantile, full=True)
DocmentTbl(rolling_quantile)

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| window_size | int |  | The size of the rolling window. |
| quantile | float |  | The quantile to compute (between 0 and 1). |
| shift | int | 1 | The number of periods to shift the data before applying the rolling quantile (default is 1). |
| min_samples | int | 1 | The minimum number of observations in the window required to have a value (default is 1). |
| **Returns** |  |  |  |

In [ ]:
#| export
class rolling_std:

    def __init__(self,
                 window_size: int,
                 shift: int = 1,
                 min_samples: int = 1):

        """
        A class to compute the rolling standard deviation of a time series with specified window size and shift.

        Parameters
        ----------
        window_size : int
            The size of the rolling window.
        shift : int, optional
            The number of periods to shift the data before applying the rolling standard deviation (default is 1).
        min_samples : int, optional
            The minimum number of observations in the window required to have a value (default is 1).
        
        Returns
        -------
        None

        """

        self.shift = shift
        self.window_size = window_size
        self.min_samples = min_samples

    def __call__(self,
                 data: Union[pd.Series, np.ndarray],
                 is_forecast: bool = False
                 ) -> pd.Series:
        """
        Compute the rolling standard deviation of the input data.
        
        Parameters
        ----------
        data : array-like
            The input time series data for which to compute the rolling standard deviation.
        is_forecast : bool, optional
            Whether the data is a forecast (default is False). If True, the shift will be adjusted to align with the forecasted period. For example, if it's a forecast, for the forecasting next value, we might want to shift by one less than usual to align with the forecasted period
        
        Returns
        -------
        pd.Series
            A Series containing the rolling standard deviation of the input data, shifted and computed according to the specified parameters.
        """
        # Return the rolling std with the specified window size and minimum samples
        if is_forecast:
            return pd.Series(data).shift(self.shift-1).rolling(self.window_size, min_periods=self.min_samples).std()
        else:
            return pd.Series(data).shift(self.shift).rolling(self.window_size, min_periods=self.min_samples).std()
    def get_name(self):
        return f"rolling_std_{self.window_size}_{self.shift}"   

In [ ]:
#| echo: false
docments(rolling_std, full=True)
DocmentTbl(rolling_std)

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| window_size | int |  | The size of the rolling window. |
| shift | int | 1 | The number of periods to shift the data before applying the rolling standard deviation (default is 1). |
| min_samples | int | 1 | The minimum number of observations in the window required to have a value (default is 1). |
| **Returns** |  |  |  |

In [ ]:
#| export

class rolling_min:
    def __init__(self,
                 window_size: int,
                 shift: int = 1,
                 min_samples: int = 1
                 ):

        """
        
        A class to compute the rolling minimum of a time series with specified window size and shift.

        Parameters
        ----------
        window_size : int
            The size of the rolling window.
        shift : int, optional
            The number of periods to shift the data before applying the rolling minimum (default is 1).
        min_samples : int, optional
            The minimum number of observations in the window required to have a value (default is 1).

        Returns
        -------
        None

        """
        self.shift = shift
        self.window_size = window_size
        self.min_samples = min_samples

    def __call__(self,
                 data: Union[pd.Series, np.ndarray],
                 is_forecast: bool = False
                 ) -> pd.Series:
        """
        Compute the rolling minimum of the input data.
        
        Parameters
        ----------
        data : array-like
            The input time series data for which to compute the rolling minimum.
        is_forecast : bool, optional
            Whether the data is a forecast (default is False). If True, the shift will be adjusted to align with the forecasted period. For example, if it's a forecast, for the forecasting next value, we might want to shift by one less than usual to align with the forecasted period

        Returns
        -------
        pd.Series
            A Series containing the rolling minimum of the input data, shifted and computed according to the specified parameters.
        """

        if is_forecast:
            # For example, if it's a forecast, for the forecasting next value, we might want to shift by one less than usual to align with the forecasted period
            return pd.Series(data).shift(self.shift-1).rolling(self.window_size, min_periods=self.min_samples).min()
        # If not a forecast, apply the usual shift
        else:
            # If not a forecast, apply the usual shift
            return pd.Series(data).shift(self.shift).rolling(self.window_size, min_periods=self.min_samples).min()
        
    def get_name(self):
        return f"rolling_min_{self.window_size}_{self.shift}"

In [ ]:
#| echo: false
docments(rolling_min, full=True)
DocmentTbl(rolling_min)

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| window_size | int |  | The size of the rolling window. |
| shift | int | 1 | The number of periods to shift the data before applying the rolling minimum (default is 1). |
| min_samples | int | 1 | The minimum number of observations in the window required to have a value (default is 1). |
| **Returns** |  |  |  |

In [ ]:
#| export

class rolling_max:
    def __init__(self,
                 window_size: int,
                 shift: int = 1,
                 min_samples: int = 1
                 ):

        """
        
        A class to compute the rolling maximum of a time series with specified window size and shift.

        Parameters
        ----------
        window_size : int
            The size of the rolling window.
        shift : int, optional
            The number of periods to shift the data before applying the rolling maximum (default is 1).
        min_samples : int, optional
            The minimum number of observations in the window required to have a value (default is 1).

        Returns
        -------
        None

        """
        self.shift = shift
        self.window_size = window_size
        self.min_samples = min_samples

    def __call__(self,
                 data: Union[pd.Series, np.ndarray],
                 is_forecast: bool = False
                 ) -> pd.Series:
        """
        Compute the rolling maximum of the input data.
        
        Parameters
        ----------
        data : array-like
            The input time series data for which to compute the rolling maximum.
        is_forecast : bool, optional
            Whether the data is a forecast (default is False). If True, the shift will be adjusted to align with the forecasted period. For example, if it's a forecast, for the forecasting next value, we might want to shift by one less than usual to align with the forecasted period

        Returns
        -------
        pd.Series
            A Series containing the rolling maximum of the input data, shifted and computed according to the specified parameters.
        """

        if is_forecast:
            # For example, if it's a forecast, for the forecasting next value, we might want to shift by one less than usual to align with the forecasted period
            return pd.Series(data).shift(self.shift-1).rolling(self.window_size, min_periods=self.min_samples).max()
        # If not a forecast, apply the usual shift
        else:
            # If not a forecast, apply the usual shift
            return pd.Series(data).shift(self.shift).rolling(self.window_size, min_periods=self.min_samples).max()
        
    def get_name(self):
        return f"rolling_max_{self.window_size}_{self.shift}"

In [ ]:
#| echo: false
docments(rolling_max, full=True)
DocmentTbl(rolling_max)

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| window_size | int |  | The size of the rolling window. |
| shift | int | 1 | The number of periods to shift the data before applying the rolling maximum (default is 1). |
| min_samples | int | 1 | The minimum number of observations in the window required to have a value (default is 1). |
| **Returns** |  |  |  |

In [ ]:
#| export

class expanding_mean:
    def __init__(self,
                 shift: int = 1
                 ):
        """
        A class to compute the expanding mean of a time series with specified shift.

        Parameters
        ----------
        shift: int
            The number of periods to shift time series data.
        
        Returns
        -------
        None
        """
        
        self.shift = shift
    def __call__(self, data, is_forecast=False):
        if is_forecast:
            return pd.Series(data).shift(self.shift-1).expanding().mean()
        else:
            return pd.Series(data).shift(self.shift).expanding().mean()
    def get_name(self):
        return f"expanding_mean_{self.shift}"

In [ ]:
#| echo: false
docments(expanding_mean, full=True)
DocmentTbl(expanding_mean)

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| shift | int | 1 |  |
| **Returns** |  |  |  |

In [ ]:
#| export

class expanding_std:
    def __init__(self,
                 shift: int = 1):
        """
        A class to compute the expanding standard deviation of a time series with specified shift.
        
        Parameters
        ----------
        shift : int, optional
            The number of periods to shift the data before applying the expanding standard deviation (default is 1).
        
        Returns
        -------
        None
        """
        self.shift = shift
    def __call__(self, data, is_forecast=False):
        if is_forecast:
            return pd.Series(data).shift(self.shift-1).expanding().std()
        else:
            return pd.Series(data).shift(self.shift).expanding().std()
    def get_name(self):
        return f"expanding_std_{self.shift}"

In [ ]:
#| echo: false
docments(expanding_std, full=True)
DocmentTbl(expanding_std)

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| shift | int | 1 | The number of periods to shift the data before applying the expanding standard deviation (default is 1). |
| **Returns** |  |  |  |

In [ ]:
#| export

class expanding_quantile:
    def __init__(self,
                 shift: int = 1,
                 quantile: float = 0.5):
        
        """
        A class to compute the expanding quantile of a time series with specified shift and quantile.
        
        Parameters
        ----------
        shift : int, optional
            The number of periods to shift the data before applying the expanding quantile (default is 1).
        quantile : float, optional
            The quantile to compute (between 0 and 1) (default is 0.5).
        
        Returns
        -------
        None
        """
        self.shift = shift
        self.quantile = quantile

    def __call__(self, data,
                 is_forecast=False):
        """
        Compute the expanding quantile.
        """
        if is_forecast:
            return pd.Series(data).shift(self.shift-1).expanding().quantile(self.quantile)
        else:
            return pd.Series(data).shift(self.shift).expanding().quantile(self.quantile)
    def get_name(self):
        return f"expanding_quantile_{self.quantile}_{self.shift}"   
        

In [ ]:
#| echo: false
docments(expanding_quantile, full=True)
DocmentTbl(expanding_quantile)

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| shift | int | 1 | The number of periods to shift the data before applying the expanding quantile (default is 1). |
| quantile | float | 0.5 | The quantile to compute (between 0 and 1) (default is 0.5). |
| **Returns** |  |  |  |